In [58]:
from pathlib import Path
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

In [59]:
# Original government CSV files
RAINFALL_FILES = [
    "datasets/raw/rainfallmanual_daily_1991_2020.csv",
    "datasets/raw/rainfall_manual_daily_2021_2025.csv"
]

DISCHARGE_FILES = [
    "datasets/raw/riverdischarge_manual_daily_1950_2000.csv",
    "datasets/raw/river_discharge_manual_daily_2001_2025.csv"
]

STATION = "Khanditar"

# Station-specific processed output directory
OUTPUT_DIR = os.path.join("datasets/processed", STATION)
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOOKBACK = 30
HORIZON = 5

TRAIN_END = "2015-12-31"
VAL_START = "2016-01-01"
VAL_END = "2018-12-31"
TEST_START = "2019-01-01"
TEST_END = "2021-11-30"

In [ ]:
# Check that the files exist
for file in RAINFALL_FILES + DISCHARGE_FILES:
    if not os.path.exists(file):
        raise FileNotFoundError(f"File not found: {file}")

# Load and combine rainfall files
rainfall = pd.concat([pd.read_csv(file) for file in RAINFALL_FILES], ignore_index=True)

# Load and combine discharge files
discharge = pd.concat([pd.read_csv(file) for file in DISCHARGE_FILES], ignore_index=True)

print("Rainfall columns:")
print(rainfall.columns.tolist())

print("\nDischarge columns:")
print(discharge.columns.tolist())

print("\nRainfall shape:", rainfall.shape)
print("Discharge shape:", discharge.shape)

display(rainfall.head())
display(discharge.head())

Rainfall columns:
['SlNo', 'Station', 'Agency', 'State LGD Code', 'State', 'District LGD Code', 'District', 'Tehsil', 'Block', 'Village', 'River', 'Basin', 'Tributary', 'Subtributary', 'SubSubtributary', 'Local River', 'Latitude', 'Longitude', 'Data Acquisition Time', 'Manual Daily Rainfall (mm)']

Discharge columns:
['SlNo', 'Station', 'Agency', 'State LGD Code', 'State', 'District LGD Code', 'District', 'Tehsil', 'Block', 'Village', 'River', 'Basin', 'Tributary', 'Subtributary', 'SubSubtributary', 'Local River', 'Latitude', 'Longitude', 'Data Acquisition Time', 'Manual Daily River Water Discharge (m3/sec)']

Rainfall shape: (91146, 20)
Discharge shape: (277270, 20)


,SlNo,Station,Agency,State LGD Code,State,District LGD Code,District,Tehsil,Block,Village,River,Basin,Tributary,Subtributary,SubSubtributary,Local River,Latitude,Longitude,Data Acquisition Time,Manual Daily Rainfall (mm)
0,1,Badapandusar,Odisha,21,Odisha,367,NAYAGARH,NAYAGARH,-,-,Mahanadi,Mahanadi,-,-,-,Kusumi,20.151111,85.180833,03-05-2001 08:30,7.4
1,2,Badapandusar,Odisha,21,Odisha,367,NAYAGARH,NAYAGARH,-,-,Mahanadi,Mahanadi,-,-,-,Kusumi,20.151111,85.180833,07-05-2001 08:30,2.2
2,3,Badapandusar,Odisha,21,Odisha,367,NAYAGARH,NAYAGARH,-,-,Mahanadi,Mahanadi,-,-,-,Kusumi,20.151111,85.180833,14-05-2001 08:30,2.2
3,4,Badapandusar,Odisha,21,Odisha,367,NAYAGARH,NAYAGARH,-,-,Mahanadi,Mahanadi,-,-,-,Kusumi,20.151111,85.180833,12-06-2001 08:30,38.0
4,5,Badapandusar,Odisha,21,Odisha,367,NAYAGARH,NAYAGARH,-,-,Mahanadi,Mahanadi,-,-,-,Kusumi,20.151111,85.180833,13-06-2001 08:30,178.0


,SlNo,Station,Agency,State LGD Code,State,District LGD Code,District,Tehsil,Block,Village,River,Basin,Tributary,Subtributary,SubSubtributary,Local River,Latitude,Longitude,Data Acquisition Time,Manual Daily River Water Discharge (m3/sec)
0,1,Badapandusar,Odisha,21,Odisha,367,NAYAGARH,NAYAGARH,-,-,Mahanadi,Mahanadi,-,-,-,Kusumi,20.151111,85.180833,01-01-2000 08:30,0.0
1,2,Badapandusar,Odisha,21,Odisha,367,NAYAGARH,NAYAGARH,-,-,Mahanadi,Mahanadi,-,-,-,Kusumi,20.151111,85.180833,02-01-2000 08:30,0.0
2,3,Badapandusar,Odisha,21,Odisha,367,NAYAGARH,NAYAGARH,-,-,Mahanadi,Mahanadi,-,-,-,Kusumi,20.151111,85.180833,03-01-2000 08:30,0.0
3,4,Badapandusar,Odisha,21,Odisha,367,NAYAGARH,NAYAGARH,-,-,Mahanadi,Mahanadi,-,-,-,Kusumi,20.151111,85.180833,04-01-2000 08:30,0.0
4,5,Badapandusar,Odisha,21,Odisha,367,NAYAGARH,NAYAGARH,-,-,Mahanadi,Mahanadi,-,-,-,Kusumi,20.151111,85.180833,05-01-2000 08:30,0.0


In [ ]:
# Convert raw columns to standard Date and numeric columns
rainfall["Date"] = pd.to_datetime(
    rainfall["Data Acquisition Time"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
).dt.floor("D")

discharge["Date"] = pd.to_datetime(
    discharge["Data Acquisition Time"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
).dt.floor("D")

# Convert measurements to numeric
rainfall["Rainfall"] = pd.to_numeric(
    rainfall["Manual Daily Rainfall (mm)"],
    errors="coerce"
)

discharge["Discharge"] = pd.to_numeric(
    discharge["Manual Daily River Water Discharge (m3/sec)"],
    errors="coerce"
)

# Select Station 
rainfall = rainfall[
    rainfall["Station"].astype(str).str.strip().str.casefold() == STATION.casefold()
].copy()

discharge = discharge[
    discharge["Station"].astype(str).str.strip().str.casefold() == STATION.casefold()
].copy()

# Sanitize known raw recording typos (Naraj August 2016 10x decimal shift)
discharge.loc[
    (discharge["Station"].astype(str).str.strip().str.casefold() == "naraj") & (discharge["Discharge"] > 50000),
    "Discharge"
] /= 10.0

# Remove invalid dates
rainfall = rainfall.dropna(subset=["Date"])
discharge = discharge.dropna(subset=["Date"])

# Aggregate multiple readings taken on the same date to daily averages
rainfall = rainfall.groupby("Date", as_index=False).agg({"Station": "first", "Rainfall": "mean"})
discharge = discharge.groupby("Date", as_index=False).agg({"Station": "first", "Discharge": "mean"})

print(f"{STATION} daily rainfall rows:", len(rainfall))
print(f"{STATION} daily discharge rows:", len(discharge))

print("Missing rainfall measurements:", rainfall["Rainfall"].isna().sum())
print("Missing discharge measurements:", discharge["Discharge"].isna().sum())

Khanditar daily rainfall rows: 1982
Khanditar daily discharge rows: 8005
Missing rainfall measurements: 0
Missing discharge measurements: 0


In [62]:
# Find duplicate rainfall dates
rain_duplicates = rainfall[
    rainfall.duplicated("Date", keep=False)
].sort_values("Date")

# Find duplicate discharge dates
flow_duplicates = discharge[
    discharge.duplicated("Date", keep=False)
].sort_values("Date")

print("Rainfall rows in duplicate groups:", len(rain_duplicates))
print("Discharge rows in duplicate groups:", len(flow_duplicates))

# Save duplicate records for inspection
rain_duplicates.to_csv(
    os.path.join(OUTPUT_DIR, "rainfall_duplicates.csv"),
    index=False
)

flow_duplicates.to_csv(
    os.path.join(OUTPUT_DIR, "discharge_duplicates.csv"),
    index=False
)

print("\nSample duplicate discharge records:")
display(flow_duplicates.head(30))

print("\nSample duplicate rainfall records:")
display(rain_duplicates.head(30))

# Do not proceed until duplicate dates have been investigated
if not rain_duplicates.empty or not flow_duplicates.empty:
    raise ValueError(
        "Duplicate dates detected. Inspect the saved duplicate CSV files "
        "and resolve the duplicates before continuing to Cell 5."
    )

Rainfall rows in duplicate groups: 0
Discharge rows in duplicate groups: 0

Sample duplicate discharge records:


,Date,Station,Discharge



Sample duplicate rainfall records:


,Date,Station,Rainfall


In [ ]:
# Keep only the required columns
rainfall_daily = rainfall[["Date", "Rainfall"]].copy()
discharge_daily = discharge[["Date", "Discharge"]].copy()

# Create the daily calendar from discharge coverage
start_date = discharge_daily["Date"].min()
end_date = discharge_daily["Date"].max()

calendar = pd.DataFrame({
    "Date": pd.date_range(start_date, end_date, freq="D")
})

# Join the observations
df = calendar.merge(discharge_daily, on="Date", how="left", validate="one_to_one")
df = df.merge(rainfall_daily, on="Date", how="left", validate="one_to_one")

# Preserve which measurements were genuinely logged in raw files
df["Discharge_observed"] = df["Discharge"].notna().astype(int)
df["Rainfall_observed"] = df["Rainfall"].notna().astype(int)

# Display dataset information
print("Total calendar days:", len(df))

print("\nDate range:")
print(df["Date"].min(), "to", df["Date"].max())

print("\nMissing measurements:")
print(df[["Discharge", "Rainfall"]].isna().sum())

print("\nObserved measurements:")
print(df[["Discharge_observed", "Rainfall_observed"]].sum())

print("\nFirst 10 rows:")
display(df.head(10))


print("Total calendar days:", len(df))
print("Missing discharge days:", df["Discharge"].isna().sum())
print("Missing rainfall days:", df["Rainfall"].isna().sum())

display(df.head())

Total calendar days: 8005

Date range:
2000-01-01 00:00:00 to 2021-11-30 00:00:00

Missing measurements:
Discharge       0
Rainfall     6023
dtype: int64

Observed measurements:
Discharge_observed    8005
Rainfall_observed     1982
dtype: int64

First 10 rows:


,Date,Discharge,Rainfall,Discharge_observed,Rainfall_observed
0,2000-01-01,255.44,NaN,1,0
1,2000-01-02,242.00,NaN,1,0
2,2000-01-03,224.09,NaN,1,0
3,2000-01-04,214.48,NaN,1,0
4,2000-01-05,277.80,NaN,1,0
5,2000-01-06,283.85,NaN,1,0
6,2000-01-07,295.71,NaN,1,0
7,2000-01-08,306.19,NaN,1,0
8,2000-01-09,294.67,NaN,1,0
9,2000-01-10,313.30,NaN,1,0


Total calendar days: 8005
Missing discharge days: 0
Missing rainfall days: 6023


,Date,Discharge,Rainfall,Discharge_observed,Rainfall_observed
0,2000-01-01,255.44,NaN,1,0
1,2000-01-02,242.00,NaN,1,0
2,2000-01-03,224.09,NaN,1,0
3,2000-01-04,214.48,NaN,1,0
4,2000-01-05,277.80,NaN,1,0


In [65]:
# Calculate it only when all three days have recorded rainfall.
df["Rain_3d"] = df["Rainfall"].rolling(
    window=3,
    min_periods=3
).sum()

# 2. Previous day's measurements
df["Discharge_lag1"] = df["Discharge"].shift(1)
df["Rain_lag1"] = df["Rainfall"].shift(1)

# 3. Seasonal feature
df["DayOfYear"] = df["Date"].dt.dayofyear

# 4. Define model input features
FEATURES = [
    "Discharge",
    "Rainfall",
    "Rain_3d",
    "Discharge_lag1",
    "Rain_lag1",
    "DayOfYear",
    "Rainfall_observed",
    "Discharge_observed"
]

# 5. Preserve discharge values for future target creation
df["Target_Discharge"] = df["Discharge"]

# 6. Identify dates with observed discharge
df["Target_Observed"] = df["Discharge"].notna()

# 7. Display the results
print("Feature engineering completed.")
print("Number of rows:", len(df))
print("Number of input features:", len(FEATURES))

display(
    df[["Date"] + FEATURES + ["Target_Discharge", "Target_Observed"]]
    .head(10)
)

Feature engineering completed.
Number of rows: 8005
Number of input features: 8


,Date,Discharge,Rainfall,Rain_3d,Discharge_lag1,Rain_lag1,DayOfYear,Rainfall_observed,Discharge_observed,Target_Discharge,Target_Observed
0,2000-01-01,255.44,NaN,NaN,NaN,NaN,1,0,1,255.44,True
1,2000-01-02,242.00,NaN,NaN,255.44,NaN,2,0,1,242.00,True
2,2000-01-03,224.09,NaN,NaN,242.00,NaN,3,0,1,224.09,True
3,2000-01-04,214.48,NaN,NaN,224.09,NaN,4,0,1,214.48,True
4,2000-01-05,277.80,NaN,NaN,214.48,NaN,5,0,1,277.80,True
5,2000-01-06,283.85,NaN,NaN,277.80,NaN,6,0,1,283.85,True
6,2000-01-07,295.71,NaN,NaN,283.85,NaN,7,0,1,295.71,True
7,2000-01-08,306.19,NaN,NaN,295.71,NaN,8,0,1,306.19,True
8,2000-01-09,294.67,NaN,NaN,306.19,NaN,9,0,1,294.67,True
9,2000-01-10,313.30,NaN,NaN,294.67,NaN,10,0,1,313.30,True


In [66]:
# Training dataset
train = df[df["Date"] <= TRAIN_END].copy()

# Validation dataset
val = df[
    (df["Date"] >= VAL_START) &
    (df["Date"] <= VAL_END)
].copy()

# Testing dataset
test = df[
    (df["Date"] >= TEST_START) &
    (df["Date"] <= TEST_END)
].copy()

# Display dataset sizes
print("Training rows:", len(train))
print("Validation rows:", len(val))
print("Testing rows:", len(test))

# Verify date ranges
print("\nTraining period:",
      train["Date"].min(), "to", train["Date"].max())

print("Validation period:",
      val["Date"].min(), "to", val["Date"].max())

print("Testing period:",
      test["Date"].min(), "to", test["Date"].max())

Training rows: 5844
Validation rows: 1096
Testing rows: 1065

Training period: 2000-01-01 00:00:00 to 2015-12-31 00:00:00
Validation period: 2016-01-01 00:00:00 to 2018-12-31 00:00:00
Testing period: 2019-01-01 00:00:00 to 2021-11-30 00:00:00


In [67]:
measurement_features = [
    "Discharge",
    "Rainfall",
    "Rain_3d",
    "Discharge_lag1",
    "Rain_lag1"
]

# Calculate medians using training data only
medians = train[measurement_features].median()

if medians.isna().any():
    raise ValueError(
        "Some training features contain no valid observations. "
        "Inspect the raw data before continuing."
    )

# Impute missing inputs using training medians
for part in [train, val, test]:
    part[measurement_features] = (
        part[measurement_features].fillna(medians)
    )

# Check missing values in all datasets
for name, part in [
    ("Training", train),
    ("Validation", val),
    ("Testing", test)
]:
    print(f"\n{name} missing values:")
    print(part[FEATURES].isna().sum())

print("\nTraining medians used for imputation:")
print(medians)


Training missing values:
Discharge             0
Rainfall              0
Rain_3d               0
Discharge_lag1        0
Rain_lag1             0
DayOfYear             0
Rainfall_observed     0
Discharge_observed    0
dtype: int64

Validation missing values:
Discharge             0
Rainfall              0
Rain_3d               0
Discharge_lag1        0
Rain_lag1             0
DayOfYear             0
Rainfall_observed     0
Discharge_observed    0
dtype: int64

Testing missing values:
Discharge             0
Rainfall              0
Rain_3d               0
Discharge_lag1        0
Rain_lag1             0
DayOfYear             0
Rainfall_observed     0
Discharge_observed    0
dtype: int64

Training medians used for imputation:
Discharge         260.51
Rainfall            8.80
Rain_3d            46.08
Discharge_lag1    260.51
Rain_lag1           8.80
dtype: float64


In [68]:
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

# Scale input features
scaler_x.fit(train[FEATURES])

# Fit target scaler using observed training discharge only
observed_train_discharge = train.loc[
    train["Target_Observed"],
    ["Target_Discharge"]
]

if observed_train_discharge.empty:
    raise ValueError("No observed discharge targets in training data.")

scaler_y.fit(observed_train_discharge)

# Transform input features
for part in [train, val, test]:
    part[FEATURES] = scaler_x.transform(part[FEATURES])

print("Input scaler fitted.")
print("Target scaler fitted.")

Input scaler fitted.
Target scaler fitted.


In [ ]:
def create_sequences(target_part, history_part=None):

    # Add the previous 30 days of historical context when provided
    if history_part is not None:
        history = history_part.tail(LOOKBACK)
        combined = pd.concat([history, target_part], ignore_index=True)
        history_length = len(history)
    else:
        combined = target_part.copy()
        history_length = 0

    # Input features are already scaled
    X_values = combined[FEATURES].to_numpy(dtype=float)

    # Keep target discharge in its original units
    target_values = combined["Target_Discharge"].to_numpy(dtype=float)

    # Track which discharge targets were actually observed
    target_observed = combined["Target_Observed"].to_numpy(dtype=bool)

    X, y = [], []

    # Generate sequences with five future days as targets
    for target_start in range(max(LOOKBACK, history_length), len(combined) - HORIZON + 1):

        # Previous 30 days
        x_seq = X_values[target_start - LOOKBACK:target_start]

        # Following 5 days
        y_seq = target_values[target_start:target_start + HORIZON]

        mask_seq = target_observed[target_start:target_start + HORIZON]

        # Skip sequences if any target discharge is missing
        if not mask_seq.all():
            continue

        # Skip sequences containing invalid numerical values
        if not np.isfinite(x_seq).all():
            continue

        if not np.isfinite(y_seq).all():
            continue

        X.append(x_seq)
        y.append(y_seq)

    # Convert lists into NumPy arrays
    X = np.asarray(X, dtype=float).reshape(-1, LOOKBACK, len(FEATURES))

    y = np.asarray(y, dtype=float).reshape(-1, HORIZON)

    # Scale the target discharge values
    if len(y) > 0:
        y = scaler_y.transform(y.reshape(-1, 1)).reshape(-1, HORIZON)

    return X, y


# Create training sequences
X_train, y_train = create_sequences(train)

# Use the last 30 training days as validation history
X_val, y_val = create_sequences(val, history_part=train)

# Use the last 30 validation days as testing history
X_test, y_test = create_sequences(test, history_part=val)

# Display the resulting shapes
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (5810, 30, 8)
y_train: (5810, 5)
X_val: (1092, 30, 8)
y_val: (1092, 5)
X_test: (1061, 30, 8)
y_test: (1061, 5)


c:\Users\ADARSH S SAHOO\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\ADARSH S SAHOO\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\ADARSH S SAHOO\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [ ]:
# Save training arrays
np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)

# Save validation arrays
np.save(os.path.join(OUTPUT_DIR, "X_val.npy"), X_val)
np.save(os.path.join(OUTPUT_DIR, "y_val.npy"), y_val)

# Save test arrays
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)

# Save scalers
joblib.dump(scaler_x, os.path.join(OUTPUT_DIR, "scaler_x.pkl"))
joblib.dump(scaler_y, os.path.join(OUTPUT_DIR, "scaler_y.pkl"))

# Save processed CSV files
train.to_csv(os.path.join(OUTPUT_DIR, "train_processed.csv"), index=False)

val.to_csv(os.path.join(OUTPUT_DIR, "val_processed.csv"), index=False)

test.to_csv(os.path.join(OUTPUT_DIR, "test_processed.csv"), index=False)

# Save metadata
metadata = {
    "station": STATION,
    "lookback_days": LOOKBACK,
    "forecast_horizon_days": HORIZON,
    "features": FEATURES,
    "target": "Discharge",
    "train_end": str(TRAIN_END),
    "validation_start": str(VAL_START),
    "validation_end": str(VAL_END),
    "test_start": str(TEST_START),
    "test_end": str(TEST_END),
    "X_train_shape": list(X_train.shape),
    "y_train_shape": list(y_train.shape),
    "X_val_shape": list(X_val.shape),
    "y_val_shape": list(y_val.shape),
    "X_test_shape": list(X_test.shape),
    "y_test_shape": list(y_test.shape),
    "processed_csv_files": [
        "train_processed.csv",
        "val_processed.csv",
        "test_processed.csv"
    ]
}

with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

print("Output directory:", OUTPUT_DIR)

print("\nFiles saved:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print("-", filename)

Output directory: datasets/processed\Khanditar

Files saved:
- X_test.npy
- X_train.npy
- X_val.npy
- discharge_duplicates.csv
- metadata.json
- rainfall_duplicates.csv
- scaler_x.pkl
- scaler_y.pkl
- test_processed.csv
- train_processed.csv
- val_processed.csv
- y_test.npy
- y_train.npy
- y_val.npy
